# Workout Form Check: Train and Test Live

This notebook trains a frame-level form classifier on the open UI-PRMD rehabilitation exercise dataset, then runs local webcam pose estimation and displays the live pose beside an example correct pose. Pose landmarks come from a pretrained Ultralytics YOLO pose model; the correct/incorrect classifier is trained below from labeled UI-PRMD skeleton sequences.

## Dataset setup
Download UI-PRMD from the [official dataset page](https://webpages.uidaho.edu/ui-prmd/) and unpack it under `data/UI-PRMD`. This dataset contains segmented 3D exercise skeletons. Dataset layouts and access terms may vary; follow the dataset's current usage terms. The loader searches for files ending in `positions.txt` with 66 values per frame (22 joints x 3 coordinates). It only trains when both correct and incorrect labels are explicit in the folder/file names or in `data/UI-PRMD/labels.csv`.

If the archive does not identify quality labels, create `labels.csv` with columns `file,label,subject_id`. `file` is a path relative to the dataset directory, `label` is `correct` or `incorrect`, and `subject_id` identifies the participant. Do not infer form labels from exercise names.

The webcam comparison is a research demo, not medical or coaching advice. Dataset and webcam camera views differ, so a trained result may not generalize. Webcam capture requires a camera device accessible to the notebook kernel; a remote dev container may not have access to your host webcam.

In [1]:
%pip install -q ultralytics scikit-learn ipywidgets joblib

from pathlib import Path
import csv
import re
import warnings

import cv2
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit

DATA_ROOT = Path('data/UI-PRMD')
LABEL_CSV = DATA_ROOT / 'labels.csv'
MODEL_PATH = Path('workout_form_classifier.joblib')
RANDOM_STATE = 42

# UI-PRMD 22-joint order: waist, spine, chest, neck, head, head tip,
# left collar/arm/forearm/hand, right collar/arm/forearm/hand,
# left hip/knee/foot/toes, right hip/knee/foot/toes.
FEATURE_NAMES = ['left_knee_angle', 'right_knee_angle', 'left_hip_angle', 'right_hip_angle', 'torso_lean']

def angle_at(first, vertex, last):
    first_vector = np.asarray(first, dtype=float) - vertex
    last_vector = np.asarray(last, dtype=float) - vertex
    denominator = np.linalg.norm(first_vector) * np.linalg.norm(last_vector)
    if denominator < 1e-8:
        return np.nan
    cosine = np.clip(np.dot(first_vector, last_vector) / denominator, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))

def ui_prmd_features(pose):
    xy = pose[:, :2]
    values = [
        angle_at(xy[14], xy[15], xy[16]),
        angle_at(xy[18], xy[19], xy[20]),
        angle_at(xy[6], xy[14], xy[15]),
        angle_at(xy[10], xy[18], xy[19]),
        np.degrees(np.arctan2(abs(xy[2, 0] - xy[0, 0]), max(abs(xy[2, 1] - xy[0, 1]), 1e-6))),
    ]
    return np.asarray(values, dtype=np.float32)

def normalize_quality_label(value):
    tokens = set(re.findall(r'[a-z]+', str(value).lower()))
    correct_tokens = {'correct', 'good', 'proper', 'ideal'}
    incorrect_tokens = {'incorrect', 'bad', 'poor', 'faulty', 'impaired'}
    correct = bool(tokens & correct_tokens)
    incorrect = bool(tokens & incorrect_tokens)
    if correct == incorrect:
        return None
    return 'correct' if correct else 'incorrect'

print('Setup ready. Dataset folder:', DATA_ROOT.resolve())

^C
Note: you may need to restart the kernel to use updated packages.


ImportError: libGL.so.1: cannot open shared object file: No such file or directory

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Download and unpack UI-PRMD under {DATA_ROOT}. See the first cell for the official source.')

manifest = {}
if LABEL_CSV.exists():
    with LABEL_CSV.open(newline='', encoding='utf-8') as handle:
        for row in csv.DictReader(handle):
            relative_file = str(row.get('file', '')).replace('\\', '/')
            label = normalize_quality_label(row.get('label', ''))
            subject_id = str(row.get('subject_id', '')).strip()
            if not relative_file or label is None or not subject_id:
                raise ValueError('labels.csv requires file, a correct/incorrect label, and subject_id for every row.')
            manifest[relative_file] = (label, subject_id)

sequence_records = []
position_files = sorted(DATA_ROOT.rglob('*positions.txt'))
if not position_files:
    raise FileNotFoundError(f'No *positions.txt files found beneath {DATA_ROOT}.')

for path in position_files:
    relative_file = path.relative_to(DATA_ROOT).as_posix()
    manifest_entry = manifest.get(relative_file)
    if manifest_entry is None:
        manifest_entry = manifest.get(path.name)
    if manifest_entry is not None:
        label, subject_id = manifest_entry
    else:
        label = normalize_quality_label(' '.join(path.parts) + ' ' + path.stem)
        subject_match = re.search(r's(\d+)', str(path), flags=re.IGNORECASE)
        if label is None or subject_match is None:
            continue
        subject_id = subject_match.group(0).lower()
    raw = np.loadtxt(path, dtype=np.float32)
    raw = np.atleast_2d(raw)
    if raw.shape[1] != 66:
        warnings.warn(f'Skipping {relative_file}: expected 66 values per frame, got {raw.shape[1]}.')
        continue
    poses = raw.reshape(-1, 22, 3)
    features = np.vstack([ui_prmd_features(frame) for frame in poses])
    valid = np.isfinite(features).all(axis=1)
    if valid.any():
        sequence_records.append({'file': relative_file, 'label': label, 'subject_id': subject_id, 'poses': poses[valid], 'features': features[valid]})

if not sequence_records:
    raise ValueError('No explicitly labeled position sequences loaded. Add labels.csv with file,label,subject_id; labels are never guessed from pose coordinates.')
labels_found = {record['label'] for record in sequence_records}
if labels_found != {'correct', 'incorrect'}:
    raise ValueError(f'Training requires both correct and incorrect sequences; found {sorted(labels_found)}. Check labels.csv and dataset labels.')
print(f'Loaded {len(sequence_records)} labeled sequences from {len({record["subject_id"] for record in sequence_records})} subjects.')
print('Sequence labels:', {label: sum(record['label'] == label for record in sequence_records) for label in sorted(labels_found)})

In [ ]:
sequence_indices = np.arange(len(sequence_records))
groups = np.asarray([record['subject_id'] for record in sequence_records])
sequence_labels = np.asarray([record['label'] for record in sequence_records])
splitter = GroupShuffleSplit(n_splits=100, test_size=0.2, random_state=RANDOM_STATE)
split = None
for train_indices, test_indices in splitter.split(sequence_indices, sequence_labels, groups):
    if set(sequence_labels[train_indices]) == {'correct', 'incorrect'} and set(sequence_labels[test_indices]) == {'correct', 'incorrect'}:
        split = (train_indices, test_indices)
        break
if split is None:
    raise ValueError('Could not create a subject-separated split containing both labels. Add more labeled subjects/sequences; never split frames from the same sequence across train and test.')
train_indices, test_indices = split

def stack_frames(indices):
    features = np.concatenate([sequence_records[index]['features'] for index in indices])
    labels = np.concatenate([np.repeat(sequence_records[index]['label'], len(sequence_records[index]['features'])) for index in indices])
    return features, labels

train_x, train_y = stack_frames(train_indices)
test_x, test_y = stack_frames(test_indices)
classifier = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
classifier.fit(train_x, train_y)

# Evaluate at the sequence level: aggregate frame predictions within each held-out sequence.
true_sequence_labels = []
predicted_sequence_labels = []
for index in test_indices:
    record = sequence_records[index]
    frame_predictions = classifier.predict(record['features'])
    counts = {label: int(np.sum(frame_predictions == label)) for label in classifier.classes_}
    predicted_sequence_labels.append(max(counts, key=counts.get))
    true_sequence_labels.append(record['label'])

print('Subject-separated held-out balanced accuracy:', round(balanced_accuracy_score(true_sequence_labels, predicted_sequence_labels), 3))
print(classification_report(true_sequence_labels, predicted_sequence_labels, zero_division=0))
print('Confusion matrix [correct, incorrect]:\n', confusion_matrix(true_sequence_labels, predicted_sequence_labels, labels=['correct', 'incorrect']))

# Use a correct training sequence as a visual reference, not as a personalized coaching prescription.
correct_train_indices = [index for index in train_indices if sequence_records[index]['label'] == 'correct']
if not correct_train_indices:
    raise ValueError('No correct-form training sequence is available for the side-by-side reference.')
reference_record = sequence_records[correct_train_indices[0]]
reference_center = np.median(reference_record['features'], axis=0)
reference_frame_index = int(np.argmin(np.linalg.norm(reference_record['features'] - reference_center, axis=1)))
REFERENCE_POSE = reference_record['poses'][reference_frame_index]
joblib.dump({'classifier': classifier, 'feature_names': FEATURE_NAMES}, MODEL_PATH)
print('Saved trained classifier to', MODEL_PATH.resolve())
print('Training subjects:', sorted(set(groups[train_indices])))
print('Held-out subjects:', sorted(set(groups[test_indices])))

## Live webcam test

Run the next cell to load YOLO pose weights and start the camera. The left panel shows the live estimated skeleton and frame-level classifier result; the right panel shows a representative correct-labeled dataset pose. Press **Start camera** / **Stop camera** to control the stream. If camera access fails in VS Code Remote, run the notebook on a machine where the Jupyter kernel can access a webcam.

The live camera uses 2D COCO body landmarks while UI-PRMD uses Kinect 3D joints. The selected joint angles are only an approximate feature match; validate across your camera setup and participants before interpreting the result.

In [ ]:
from threading import Thread
import time
import ipywidgets as widgets
from IPython.display import display
from ultralytics import YOLO

pose_model = YOLO('yolo11n-pose.pt')
CAMERA_INDEX = 0
MIN_KEYPOINT_CONFIDENCE = 0.25
MIN_CLASSIFIER_CONFIDENCE = 0.70


def coco_pose_features(points, confidence):
    required = [5, 6, 11, 12, 13, 14, 15, 16]
    if confidence is None or np.any(confidence[required] < MIN_KEYPOINT_CONFIDENCE):
        return None
    xy = points[:, :2]
    shoulder_center = (xy[5] + xy[6]) / 2.0
    hip_center = (xy[11] + xy[12]) / 2.0
    torso = np.degrees(np.arctan2(abs(shoulder_center[0] - hip_center[0]), max(abs(shoulder_center[1] - hip_center[1]), 1e-6)))
    return np.asarray([
        angle_at(xy[11], xy[13], xy[15]),
        angle_at(xy[12], xy[14], xy[16]),
        angle_at(xy[5], xy[11], xy[13]),
        angle_at(xy[6], xy[12], xy[14]),
        torso,
    ], dtype=np.float32)


def inspect_live_issues(points, confidence):
    required = [5, 6, 11, 12, 13, 14, 15, 16]
    if confidence is None or np.any(confidence[required] < MIN_KEYPOINT_CONFIDENCE):
        return None, {}
    xy = points[:, :2]
    left_knee = angle_at(xy[11], xy[13], xy[15])
    right_knee = angle_at(xy[12], xy[14], xy[16])
    hip_width = np.linalg.norm(xy[11] - xy[12])
    ankle_width = np.linalg.norm(xy[15] - xy[16])
    knee_width = np.linalg.norm(xy[13] - xy[14])
    expected_width = (hip_width + ankle_width) / 2
    knee_ratio = knee_width / expected_width if expected_width > 1 else 1.0
    shoulder_center = (xy[5] + xy[6]) / 2
    hip_center = (xy[11] + xy[12]) / 2
    torso_lean = np.degrees(np.arctan2(abs(shoulder_center[0] - hip_center[0]), max(abs(shoulder_center[1] - hip_center[1]), 1e-6)))
    issues = []
    if knee_ratio < 0.62:
        issues.append('Knees may be moving inward')
    if np.isfinite(left_knee) and np.isfinite(right_knee) and abs(left_knee - right_knee) > 30:
        issues.append('Knee bend differs side to side')
    if torso_lean > 50:
        issues.append('Substantial torso lean')
    metrics = {
        'knee width ratio': round(float(knee_ratio), 2),
        'left knee angle': round(float(left_knee), 1) if np.isfinite(left_knee) else None,
        'right knee angle': round(float(right_knee), 1) if np.isfinite(right_knee) else None,
        'torso lean degrees': round(float(torso_lean), 1),
    }
    return issues, metrics


def make_reference_image(pose):
    canvas = np.full((420, 320, 3), 248, dtype=np.uint8)
    xy = pose[:, :2].astype(float)
    low, high = xy.min(axis=0), xy.max(axis=0)
    span = np.maximum(high - low, 1e-6)
    scale = min(250 / span[0], 350 / span[1])
    points = ((xy - low) * scale).astype(int)
    points[:, 0] += int((320 - span[0] * scale) / 2)
    points[:, 1] += int((420 - span[1] * scale) / 2)
    bones = [(0, 1), (1, 2), (2, 3), (3, 4), (2, 6), (6, 7), (7, 8), (8, 9), (2, 10), (10, 11), (11, 12), (12, 13), (0, 14), (14, 15), (15, 16), (16, 17), (0, 18), (18, 19), (19, 20), (20, 21)]
    for first, last in bones:
        cv2.line(canvas, tuple(points[first]), tuple(points[last]), (75, 125, 35), 3, cv2.LINE_AA)
    for point in points:
        cv2.circle(canvas, tuple(point), 5, (40, 80, 30), -1, cv2.LINE_AA)
    return canvas


def to_jpeg(image):
    success, encoded = cv2.imencode('.jpg', image, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
    return encoded.tobytes() if success else b''


live_image = widgets.Image(format='jpeg', width=480, height=360)
reference_image = widgets.Image(value=to_jpeg(make_reference_image(REFERENCE_POSE)), format='jpeg', width=320, height=420)
status = widgets.HTML(value='<b>Camera stopped.</b>')
camera_button = widgets.Button(description='Start camera', button_style='success', icon='play')
live_state = {'running': False}


def camera_loop():
    capture = cv2.VideoCapture(CAMERA_INDEX)
    if not capture.isOpened():
        live_state['running'] = False
        status.value = '<b style="color:#b23b32">Could not open webcam. Check kernel camera access and CAMERA_INDEX.</b>'
        camera_button.description = 'Start camera'
        camera_button.button_style = 'success'
        capture.release()
        return
    status.value = '<b>Camera live.</b> Stand back so your full body is visible.'
    try:
        while live_state['running']:
            ok, frame = capture.read()
            if not ok:
                status.value = '<b style="color:#b23b32">Camera frame read failed.</b>'
                break
            result = pose_model.predict(frame, verbose=False, conf=0.35)[0]
            annotated = result.plot()
            verdict_text = 'Pose not detected'
            issue_text = 'No person pose detected'
            issue_color = (40, 170, 190)
            color = (40, 170, 190)
            if result.keypoints is not None and len(result.keypoints.xy):
                person_index = int(result.boxes.conf.argmax().item()) if result.boxes is not None and len(result.boxes) else 0
                points = result.keypoints.xy[person_index].cpu().numpy()
                confidence = result.keypoints.conf[person_index].cpu().numpy() if result.keypoints.conf is not None else None
                features = coco_pose_features(points, confidence)
                issues, metrics = inspect_live_issues(points, confidence)
                if issues is None:
                    issue_text = 'Move into full view for issue checks'
                    issue_color = (30, 160, 210)
                elif issues:
                    issue_text = 'Possible: ' + ', '.join(issues)
                    issue_color = (50, 70, 210)
                else:
                    issue_text = 'No configured issue cue detected'
                    issue_color = (60, 145, 40)
                if features is not None and np.isfinite(features).all():
                    probabilities = classifier.predict_proba(features.reshape(1, -1))[0]
                    best_index = int(np.argmax(probabilities))
                    predicted = str(classifier.classes_[best_index])
                    certainty = float(probabilities[best_index])
                    verdict_text = predicted.title() if certainty >= MIN_CLASSIFIER_CONFIDENCE else 'Unclear pose'
                    color = (60, 165, 45) if predicted == 'correct' else (50, 70, 210)
                    if certainty < MIN_CLASSIFIER_CONFIDENCE:
                        color = (30, 160, 210)
                    verdict_text += f' | {certainty:.0%}'
                    issue_text += ' | ' + ', '.join(f'{key}: {value}' for key, value in metrics.items())
                else:
                    verdict_text = 'Move into full view'
            cv2.putText(annotated, verdict_text, (18, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
            cv2.putText(annotated, issue_text[:100], (18, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.55, issue_color, 2, cv2.LINE_AA)
            live_image.value = to_jpeg(annotated)
            status.value = f'<b>Camera live.</b> {issue_text}'
            time.sleep(0.03)
    finally:
        capture.release()
        live_state['running'] = False
        camera_button.description = 'Start camera'
        camera_button.button_style = 'success'
        if 'Could not open' not in status.value:
            status.value = '<b>Camera stopped.</b>'


def toggle_camera(_):
    if live_state['running']:
        live_state['running'] = False
        camera_button.description = 'Stopping...'
        camera_button.button_style = 'warning'
        return
    live_state['running'] = True
    camera_button.description = 'Stop camera'
    camera_button.button_style = 'danger'
    Thread(target=camera_loop, daemon=True).start()


camera_button.on_click(toggle_camera)
panels = widgets.HBox([
    widgets.VBox([widgets.HTML('<b>Live camera pose + form issues</b>'), live_image]),
    widgets.VBox([widgets.HTML('<b>Correct reference pose</b>'), reference_image]),
], layout=widgets.Layout(align_items='flex-start', gap='16px'))
display(widgets.VBox([status, camera_button, panels]))

In [ ]:
%pip install -q yt-dlp

from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.parse import urlparse
from IPython.display import Image as DisplayImage, clear_output, display
import yt_dlp

MAX_YOUTUBE_SECONDS = 300
YOUTUBE_SAMPLE_INTERVAL = 1.0
MAX_YOUTUBE_SAMPLES = 12

youtube_url = widgets.Text(
    description='YouTube URL',
    placeholder='https://www.youtube.com/watch?v=...',
    layout=widgets.Layout(width='75%'),
)
youtube_button = widgets.Button(description='Analyze video', button_style='primary', icon='play')
youtube_status = widgets.HTML(value='Paste an exercise video link. Only videos up to 5 minutes are accepted.')
youtube_output = widgets.Output()

def analyze_youtube_video(_):
    host = (urlparse(youtube_url.value.strip()).hostname or '').lower()
    if host not in {'youtube.com', 'www.youtube.com', 'm.youtube.com', 'youtu.be', 'www.youtube-nocookie.com'}:
        youtube_status.value = '<b style="color:#b23b32">Enter a valid YouTube video URL.</b>'
        return

    youtube_button.disabled = True
    youtube_status.value = '<b>Fetching video and analyzing sampled frames...</b>'
    with youtube_output:
        clear_output(wait=True)
        try:
            with TemporaryDirectory(prefix='form-check-') as temporary_directory:
                options = {
                    'format': 'best[height<=480][ext=mp4]/best[height<=480]/best',
                    'outtmpl': str(Path(temporary_directory) / 'exercise.%(ext)s'),
                    'noplaylist': True,
                    'quiet': True,
                    'no_warnings': True,
                    'match_filter': lambda info, *, incomplete: 'Video must be 5 minutes or shorter.' if info.get('duration') is None or info['duration'] > MAX_YOUTUBE_SECONDS else None,
                }
                with yt_dlp.YoutubeDL(options) as downloader:
                    info = downloader.extract_info(youtube_url.value.strip(), download=True)

                downloaded_files = [path for path in Path(temporary_directory).glob('exercise.*') if path.suffix.lower() in {'.mp4', '.mkv', '.webm', '.mov'}]
                if not downloaded_files:
                    raise RuntimeError('No video file was downloaded. Check the link, length limit, and YouTube access.')
                capture = cv2.VideoCapture(str(downloaded_files[0]))
                fps = capture.get(cv2.CAP_PROP_FPS) or 0
                if not capture.isOpened() or fps <= 0:
                    capture.release()
                    raise RuntimeError('Could not decode the downloaded video.')

                frame_step = max(1, round(fps * YOUTUBE_SAMPLE_INTERVAL))
                frame_index = 0
                samples = []
                while len(samples) < MAX_YOUTUBE_SAMPLES:
                    ok, frame = capture.read()
                    if not ok:
                        break
                    if frame_index % frame_step == 0:
                        result = pose_model.predict(frame, verbose=False, conf=0.35)[0]
                        annotated = result.plot()
                        verdict_text = 'Pose not detected'
                        color = (30, 160, 210)
                        if result.keypoints is not None and len(result.keypoints.xy):
                            person_index = int(result.boxes.conf.argmax().item()) if result.boxes is not None and len(result.boxes) else 0
                            points = result.keypoints.xy[person_index].cpu().numpy()
                            confidence = result.keypoints.conf[person_index].cpu().numpy() if result.keypoints.conf is not None else None
                            features = coco_pose_features(points, confidence)
                            if features is not None and np.isfinite(features).all():
                                probabilities = classifier.predict_proba(features.reshape(1, -1))[0]
                                best_index = int(np.argmax(probabilities))
                                predicted = str(classifier.classes_[best_index])
                                certainty = float(probabilities[best_index])
                                verdict_text = predicted.title() if certainty >= MIN_CLASSIFIER_CONFIDENCE else 'Unclear pose'
                                verdict_text += f' | {certainty:.0%}'
                                color = (60, 165, 45) if predicted == 'correct' else (50, 70, 210)
                                if certainty < MIN_CLASSIFIER_CONFIDENCE:
                                    color = (30, 160, 210)
                        cv2.putText(annotated, verdict_text, (18, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
                        live_panel = cv2.resize(annotated, (480, 360))
                        reference_panel = cv2.resize(make_reference_image(REFERENCE_POSE), (274, 360))
                        comparison = cv2.hconcat([live_panel, reference_panel])
                        samples.append((frame_index / fps, verdict_text, to_jpeg(comparison)))
                    frame_index += 1
                capture.release()

                if not samples:
                    raise RuntimeError('No readable frames were found in the linked video.')
                title = info.get('title', 'Exercise video')
                print(f'{title} | sampled {len(samples)} frames')
                for timestamp, verdict, jpeg in samples:
                    display(widgets.HTML(value=f'<b>{timestamp:.1f}s</b> - {verdict}'))
                    display(DisplayImage(data=jpeg))
                youtube_status.value = '<b>Analysis complete.</b> Results are approximate frame-level estimates.'
        except Exception as error:
            youtube_status.value = f'<b style="color:#b23b32">Could not analyze this link:</b> {type(error).__name__}: {error}'
        finally:
            youtube_button.disabled = False

youtube_button.on_click(analyze_youtube_video)
display(widgets.VBox([
    widgets.HTML('<h3>Exercise video from YouTube</h3><p>Analyze a short video you are authorized to use. Frames are sampled at 1-second intervals and shown beside the correct reference pose.</p>'),
    widgets.HBox([youtube_url, youtube_button]),
    youtube_status,
    youtube_output,
]))